# TransFit 中文教程

这个 notebook 只保留最常用的流程：
1. 查看模型参数
2. 直接画理论光变
3. 拟合测光和多波段数据
4. 绘图并保存链


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = None
for cand in (cwd, cwd.parent):
    if (cand / "transfit").is_dir():
        repo_root = cand
        break
if repo_root is None:
    raise RuntimeError("Could not locate local transfit package root.")

sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import transfit as tf

DATA_DIR = repo_root / "examples" / "data"
plt.rcParams["figure.dpi"] = 120

print("repo root:", repo_root)
print("data dir:", DATA_DIR)


## 查看模型参数

开始前可以先查看模型参数顺序：


In [ ]:
print("nickel :", tf.model_param_names("nickel"))
print("magnetar:", tf.model_param_names("magnetar"))
print("magnetar_ni:", tf.model_param_names("magnetar_ni"))

tf.param_template("nickel")


## 1. 画理论光变

先演示测光光变，再演示多波段光变。


In [ ]:
params_nickel = {
    "M_ej": 3.0,
    "v_ej": 1.0,
    "E_Th_in": 1.5,
    "M_Ni": 0.08,
    "R_0": 120.0,
    "x_Ni": 0.2,
    "kappa": 0.12,
    "kappa_gamma": 0.03,
    "T_floor": 4500.0,
}

bol = tf.lightcurve_bol(
    model="nickel",
    params=params_nickel,
    z=0.001728,
    t_max_days=180.0,
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(bol.t_days, bol.Lbol, color="#0b4f6c", lw=2.0)
ax.set_yscale("log")
ax.set_xlabel("Observer time (days)")
ax.set_ylabel("Bolometric luminosity (erg s$^{-1}$)")
ax.set_title("Bolometric Light Curve")
plt.show()


In [ ]:
filters = {
    "B": "johnson_cousins.B",
    "V": "johnson_cousins.V",
    "R": "johnson_cousins.R",
    "I": "johnson_cousins.I",
}

params_nickel_mb = {
    "M_ej": 3.0,
    "v_ej": 1.0,
    "E_Th_in": 1.5,
    "M_Ni": 0.08,
    "R_0": 120.0,
    "x_Ni": 0.2,
    "kappa": 0.12,
    "kappa_gamma": 0.03,
    "T_floor": 3000.0,
}

extinction = {"mw": {"ebv": 0.04, "rv": 3.1, "law": "odonnell94"}}

mb = tf.lightcurve_multiband(
    model="nickel",
    params=params_nickel_mb,
    z=0.001728,
    DL_Mpc=9.3,
    filters=filters,
    bands=["B", "V", "R", "I"],
    y_kind="mag",
    mag_system="vega",
    extinction=extinction,
    t_max_days=180.0,
)

fig, ax = plt.subplots(figsize=(8, 5.2))
for b in mb.bands:
    ax.plot(mb.t_days, mb.y[b], lw=2.0, label=b)
ax.invert_yaxis()
ax.set_xlabel("Observer time (days)")
ax.set_ylabel("Vega magnitude")
ax.set_title("Multi-band Light Curve")
ax.legend(frameon=False, ncol=4)
plt.show()


## 2. 拟合数据

下面分别演示测光拟合和多波段拟合。
为了让教程更快跑完，这里使用较短的链长。


In [ ]:
arr = np.loadtxt(DATA_DIR / "sn1993j_lbol.txt")
t_days = arr[:, 0] - np.min(arr[:, 0])

data_bol = tf.BolometricData(
    t_days=t_days,
    y=arr[:, 1],
    yerr=arr[:, 2],
)

res_bol = tf.fit_bol(
    data=data_bol,
    model="nickel",
    z=0.001728,
    priors={
        "M_ej": (0.7, 5.0),
        "v_ej": (0.2, 3.0),
        "E_Th_in": (1.0, 10.0),
        "M_Ni": (0.01, 0.2),
        "R_0": (100.0, 1200.0),
        "x_Ni": (0.1, 0.9),
        "t_shift": (-5.0, 5.0),
    },
    fixed={
        "kappa": 0.06,
        "kappa_gamma": 0.03,
    },
    sampler="emcee",
    sampler_kwargs={
        "nwalkers": 24,
        "nsteps": 120,
        "burnin": 40,
        "thin": 2,
        "seed": 123,
        "progress": False,
    },
)

print("测光 samples 形状:", res_bol.samples.shape)
print("测光最佳参数:", res_bol.best_params)


In [ ]:
df = pd.read_csv(DATA_DIR / "sn2007gr.csv")
t0 = float(np.nanmin(df["JD"].to_numpy(float)))
df["t_days"] = df["JD"].to_numpy(float) - t0

rows = []
for band_name, ycol, ecol in [
    ("B", "Bmag", "e_Bmag"),
    ("V", "Vmag", "e_Vmag"),
    ("R", "Rmag", "e_Rmag"),
    ("I", "Imag", "e_Imag"),
]:
    m = df[ycol].notna() & df[ecol].notna()
    rows.append(
        pd.DataFrame(
            {
                "t_days": df.loc[m, "t_days"].to_numpy(float),
                "band": band_name,
                "y": df.loc[m, ycol].to_numpy(float),
                "yerr": df.loc[m, ecol].to_numpy(float),
            }
        )
    )

long_df = pd.concat(rows, ignore_index=True).sort_values(["t_days", "band"]).reset_index(drop=True)

data_mb = tf.MultiBandData(
    t_days=long_df["t_days"].to_numpy(float),
    band=long_df["band"].to_numpy(str),
    y=long_df["y"].to_numpy(float),
    yerr=long_df["yerr"].to_numpy(float),
)

res_mb = tf.fit_multiband(
    data=data_mb,
    model="nickel",
    z=0.001728,
    DL_Mpc=9.3,
    filters=filters,
    y_kind="mag",
    mag_system="vega",
    extinction=extinction,
    priors={
        "M_ej": (1.0, 5.0),
        "v_ej": (0.3, 3.0),
        "E_Th_in": (0.05, 8.0),
        "M_Ni": (0.01, 0.5),
        "R_0": (10.0, 400.0),
        "T_floor": (3000.0, 8000.0),
    },
    fixed={"kappa": 0.06},
    sampler="emcee",
    sampler_kwargs={
        "nwalkers": 24,
        "nsteps": 120,
        "burnin": 40,
        "thin": 2,
        "seed": 456,
        "progress": False,
    },
)

print("多波段 samples 形状:", res_mb.samples.shape)
print("多波段最佳参数:", res_mb.best_params)


## 3. 画拟合结果

可以直接使用内置绘图函数查看拟合曲线和角图。


In [ ]:
fig_bol = tf.plot.fit_bol(
    res_bol,
    data=data_bol,
    show_1sigma=True,
    n_draws=200,
    t_pad=60.0,
)
fig_bol

fig_mb = tf.plot.fit_multiband(
    res_mb,
    data=data_mb,
    show_1sigma=True,
    n_draws=200,
    t_pad=40.0,
)
fig_mb

try:
    fig_corner = tf.plot.corner(res_mb, max_points=10000)
    fig_corner
except ImportError as e:
    print(e)
    print("如需角图，可安装: pip install corner")


## 4. 保存与读取链

`tf.save(...)` 会把样本链、对数后验、模型上下文和元数据一起保存到 `.npz` 文件里。


In [ ]:
out_path = tf.save(res_mb, path=Path("mcmc_out") / "fit_nickel_multiband_demo.npz")
loaded = tf.load(out_path)

print("保存路径:", out_path)
print("读取后的 model:", loaded["model"])
print("读取后的 sampler:", loaded["sampler"])
print("读取后的 samples 形状:", loaded["samples"].shape)
